# Refuerzo: Regresión Aplicada — Regularización, Ensambles y Tuning
## Ingeniería del Conocimiento (ISO56B) — UNCP
### Msc. Jaime Antonio Huaytalla Pariona — Semestre 2026-II

**Objetivo:** Aplicar técnicas avanzadas de regresión: regularización (Ridge, Lasso), ensambles (Random Forest, Gradient Boosting), pipelines, validación cruzada y selección de modelos.

**Nivel:** Intermedio → Avanzado

**Contenido:**
1. Estandarización y Pipelines
2. Ridge, Lasso y Elastic Net
3. Random Forest Regressor
4. Gradient Boosting Regressor
5. Validación Cruzada y Comparación
6. GridSearchCV — Optimización de hiperparámetros
7. Importancia de features

## 1. Configuración y Datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Cargar y dividir
data = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)
feature_names = data.feature_names

print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")
print(f"Features: {feature_names}")
print("✅ Datos listos")


## 2. Estandarización y Pipelines

### ¿Por qué estandarizar?

Los features tienen escalas muy diferentes:
- MedInc: 0-15 (ingreso medio)
- AveRooms: 0-140 (habitaciones promedio)
- Population: 3-35682

Los modelos con regularización (Ridge, Lasso) penalizan los coeficientes. Si un feature tiene escala grande, su coeficiente será pequeño y la penalización será desigual.

**La estandarización** transforma cada feature: z = (x - μ) / σ → media=0, std=1


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

# Comparar con y sin estandarización
lr_sin = LinearRegression().fit(X_train, y_train)
lr_con = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)

print("Coeficientes SIN estandarizar:")
for name, coef in zip(feature_names, lr_sin.coef_):
    print(f"  {name:15s}: {coef:+.6f}")

print(f"\nCoeficientes CON estandarizar (misma escala → comparables):")
for name, coef in zip(feature_names, lr_con.named_steps['linearregression'].coef_):
    print(f"  {name:15s}: {coef:+.6f}")

print(f"\n→ Para Regresión Lineal sin regularización, el R² es igual:")
print(f"  Sin std: {lr_sin.score(X_test, y_test):.4f}")
print(f"  Con std: {lr_con.score(X_test, y_test):.4f}")
print(f"→ PERO para Ridge/Lasso, la estandarización SÍ cambia los resultados.")


### Pipeline: Evitar Data Leakage

Un **pipeline** encadena pasos y garantiza que:
- El scaler se ajusta SOLO con datos de train
- Se aplica consistentemente a train y test
- No hay fuga de información

```python
# ❌ INCORRECTO
scaler.fit_transform(X)  # ve datos de test
X_train, X_test = split(X_scaled)

# ✅ CORRECTO
pipe = make_pipeline(StandardScaler(), Ridge())
pipe.fit(X_train, y_train)  # scaler aprende solo de train
```


## 3. Regularización: Ridge, Lasso y Elastic Net

La regularización **penaliza coeficientes grandes** para evitar overfitting.


In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# Comparar efecto de alpha en Ridge
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
ridge_coefs = []

fig, ax = plt.subplots(figsize=(12, 6))

for alpha in alphas:
    pipe = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    pipe.fit(X_train, y_train)
    coefs = pipe.named_steps['ridge'].coef_
    ridge_coefs.append(coefs)

ridge_coefs = np.array(ridge_coefs)

for i, name in enumerate(feature_names):
    ax.plot(np.log10(alphas), ridge_coefs[:, i], 'o-', label=name, linewidth=2)

ax.set_xlabel('log₁₀(alpha)')
ax.set_ylabel('Valor del coeficiente')
ax.set_title('Ridge: Efecto de alpha sobre los coeficientes')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observación: a medida que alpha crece, los coeficientes se ENCOGEN hacia 0")
print("pero NUNCA llegan exactamente a 0 en Ridge.")


In [ ]:
# Lasso: selección de features (coeficientes = 0)
print("LASSO — Selección de Features")
print("=" * 60)

for alpha in [0.001, 0.01, 0.1, 0.5, 1.0]:
    pipe = make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=10000))
    pipe.fit(X_train, y_train)
    coefs = pipe.named_steps['lasso'].coef_
    n_zero = sum(coefs == 0)
    r2 = pipe.score(X_test, y_test)
    selected = [name for name, c in zip(feature_names, coefs) if c != 0]
    print(f"\nalpha={alpha:6.3f} → {n_zero} features eliminados | R²={r2:.4f}")
    print(f"  Features activos: {selected}")

print("\n→ Lasso elimina features irrelevantes automáticamente.")
print("→ Alpha controla cuántos features se eliminan.")


In [ ]:
# Comparación visual: Ridge vs Lasso vs Elastic Net
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

modelos_reg = {
    'OLS (sin reg.)': make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge (α=1)':    make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Lasso (α=0.01)': make_pipeline(StandardScaler(), Lasso(alpha=0.01, max_iter=10000)),
    'Elastic Net':    make_pipeline(StandardScaler(), ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000)),
}

print(f"{'Modelo':<22s} {'R² Test':>10s} {'# Coefs ≠ 0':>14s}")
print("=" * 50)

for nombre, pipe in modelos_reg.items():
    pipe.fit(X_train, y_train)
    r2 = pipe.score(X_test, y_test)
    step_name = list(pipe.named_steps.keys())[-1]
    coefs = pipe.named_steps[step_name].coef_
    n_active = sum(coefs != 0)
    print(f"{nombre:<22s} {r2:10.4f} {n_active:>10d}/{len(coefs)}")


## 4. Modelos de Ensamble

Los ensambles combinan **múltiples modelos** para obtener mejores predicciones.

### 4.1 Random Forest
- Entrena muchos árboles con diferentes subconjuntos de datos y features
- Predicción = promedio de todos los árboles
- Reduce overfitting respecto a un solo árbol

### 4.2 Gradient Boosting
- Entrena árboles **secuencialmente**: cada uno corrige los errores del anterior
- Generalmente más preciso, pero más lento y sensible a hiperparámetros


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_r2 = rf.score(X_test, y_test)
print(f"Random Forest   → R² Test = {rf_r2:.4f}")

# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                                max_depth=4, random_state=42)
gb.fit(X_train, y_train)
gb_r2 = gb.score(X_test, y_test)
print(f"Gradient Boost  → R² Test = {gb_r2:.4f}")

# Comparar con Regresión Lineal
lr = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
lr_r2 = lr.score(X_test, y_test)
print(f"Regresión Lineal → R² Test = {lr_r2:.4f}")

print(f"\n→ Los ensambles capturan relaciones no lineales y son más precisos.")


In [ ]:
# Efecto del número de árboles en Random Forest
n_trees_list = [1, 5, 10, 25, 50, 100, 200, 300]
rf_scores = []

for n in n_trees_list:
    rf_temp = RandomForestRegressor(n_estimators=n, random_state=42, n_jobs=-1)
    rf_temp.fit(X_train, y_train)
    rf_scores.append(rf_temp.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(n_trees_list, rf_scores, 'o-', color='#1B5E20', linewidth=2, markersize=8)
ax.set_xlabel('Número de Árboles')
ax.set_ylabel('R² Test')
ax.set_title('Random Forest: Efecto del Número de Árboles')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("→ Más árboles mejora el rendimiento hasta estabilizarse (~100-200)")
print("→ A diferencia de un solo árbol, más árboles NO causan overfitting")


## 5. Validación Cruzada — Comparación Justa de Modelos

Un solo train/test split puede dar resultados sesgados. La **validación cruzada** evalúa el modelo K veces con diferentes particiones.


In [ ]:
from sklearn.model_selection import cross_val_score

# Todos los modelos en pipelines
modelos_todos = {
    'Reg. Lineal':    make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge':          make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Lasso':          make_pipeline(StandardScaler(), Lasso(alpha=0.01, max_iter=10000)),
    'Árbol (d=8)':    DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest':  RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boost': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1,
                                                 max_depth=4, random_state=42),
}

print("VALIDACIÓN CRUZADA (5-Fold) — scoring: neg_root_mean_squared_error")
print("=" * 65)
cv_results = {}

for nombre, modelo in modelos_todos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=5,
                              scoring='neg_root_mean_squared_error')
    rmse_mean = -scores.mean()
    rmse_std  = scores.std()
    cv_results[nombre] = {'RMSE_mean': rmse_mean, 'RMSE_std': rmse_std}
    print(f"{nombre:<18s} → RMSE = {rmse_mean:.4f} ± {rmse_std:.4f}")


In [ ]:
# Boxplot comparativo
from sklearn.model_selection import cross_val_score

fig, ax = plt.subplots(figsize=(12, 6))
all_scores = []
labels = []

for nombre, modelo in modelos_todos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=5,
                              scoring='neg_root_mean_squared_error')
    all_scores.append(-scores)
    labels.append(nombre)

bp = ax.boxplot(all_scores, labels=labels, patch_artist=True)
colors = ['#2E86AB', '#A23B72', '#F18F01', '#1B5E20', '#4CAF50', '#FF5722']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('RMSE (menor = mejor)')
ax.set_title('Comparación de Modelos — Cross-Validation 5-Fold')
ax.tick_params(axis='x', rotation=15)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("→ Los ensambles (Random Forest, Gradient Boosting) son superiores")
print("→ La variabilidad entre folds es baja → resultados estables")


## 6. GridSearchCV — Encontrar los Mejores Hiperparámetros

Gradient Boosting tiene varios hiperparámetros. GridSearchCV prueba combinaciones y elige la mejor usando cross-validation.


In [ ]:
from sklearn.model_selection import GridSearchCV

# Grid de hiperparámetros para Gradient Boosting
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
}

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
gb_grid.fit(X_train, y_train)

print("Mejores hiperparámetros encontrados:")
for param, val in gb_grid.best_params_.items():
    print(f"  {param}: {val}")

print(f"\nMejor RMSE (CV): {-gb_grid.best_score_:.4f}")
print(f"RMSE en Test:     {np.sqrt(mean_squared_error(y_test, gb_grid.predict(X_test))):.4f}")
print(f"R² en Test:       {gb_grid.score(X_test, y_test):.4f}")


In [ ]:
# Heatmap de resultados: learning_rate vs max_depth (con n_estimators = mejor)
from sklearn.metrics import mean_squared_error

results_df = pd.DataFrame(gb_grid.cv_results_)

best_n = gb_grid.best_params_['n_estimators']
mask = results_df['param_n_estimators'] == best_n

pivot = results_df[mask].pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_learning_rate'
)
pivot = -pivot  # Convertir a RMSE positivo

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd_r', ax=ax)
ax.set_title(f'RMSE por Combinación de Hiperparámetros (n_estimators={best_n})')
ax.set_ylabel('max_depth')
ax.set_xlabel('learning_rate')
plt.tight_layout()
plt.show()

print("→ Colores más claros = menor RMSE = mejor rendimiento")


## 7. Importancia de Features

¿Qué features contribuyen más a la predicción? Veamos tres métodos diferentes.


In [ ]:
from sklearn.inspection import permutation_importance

# Entrenar modelos finales
rf_final = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_final.fit(X_train, y_train)

ridge_final = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge_final.fit(X_train, y_train)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Coeficientes Ridge (estandarizados)
coefs = np.abs(ridge_final.named_steps['ridge'].coef_)
sorted_idx = np.argsort(coefs)
axes[0].barh(np.array(feature_names)[sorted_idx], coefs[sorted_idx], color='#2E86AB')
axes[0].set_title('Ridge: |Coeficientes|')
axes[0].set_xlabel('Valor absoluto')

# 2. Feature Importance de Random Forest (MDI)
importances = rf_final.feature_importances_
sorted_idx = np.argsort(importances)
axes[1].barh(np.array(feature_names)[sorted_idx], importances[sorted_idx], color='#1B5E20')
axes[1].set_title('Random Forest: Feature Importance (MDI)')
axes[1].set_xlabel('Importancia')

# 3. Permutation Importance
perm_imp = permutation_importance(rf_final, X_test, y_test,
                                   n_repeats=10, random_state=42, n_jobs=-1)
sorted_idx = np.argsort(perm_imp.importances_mean)
axes[2].barh(np.array(feature_names)[sorted_idx],
             perm_imp.importances_mean[sorted_idx], color='#A23B72')
axes[2].set_title('Permutation Importance')
axes[2].set_xlabel('Caída en R² al permutar')

plt.suptitle('Importancia de Features — Tres Métodos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Los 3 métodos coinciden: MedInc (ingreso medio) es el feature más importante.")
print("Esto tiene sentido intuitivo: el ingreso de una zona es el mejor predictor del precio.")


## 8. Resumen Final

### Lo que aprendimos:

| Técnica | Propósito |
|---------|----------|
| **Estandarización + Pipeline** | Preprocesamiento correcto sin data leakage |
| **Ridge (L2)** | Regularización que encoge coeficientes |
| **Lasso (L1)** | Regularización que ELIMINA features |
| **Elastic Net** | Combinación L1+L2 |
| **Random Forest** | Ensamble robusto de árboles |
| **Gradient Boosting** | Ensamble secuencial (árboles corrigen errores) |
| **Cross-Validation** | Evaluación justa y estable |
| **GridSearchCV** | Encontrar mejores hiperparámetros automáticamente |
| **Feature Importance** | Entender qué features importan |

### Flujo recomendado para cualquier problema de regresión:
1. Explorar datos (distribuciones, correlaciones, outliers)
2. Preprocesar (estandarizar si es necesario, manejar NaN)
3. Probar modelos baseline (Lineal, Ridge)
4. Probar modelos más complejos (Random Forest, Gradient Boosting)
5. Comparar con Cross-Validation
6. Optimizar el mejor con GridSearchCV
7. Evaluar modelo final en test
8. Interpretar (importancia de features)

---
*Material de refuerzo — Ingeniería del Conocimiento (ISO56B) — UNCP*
